# Roboat A/B training on Colab (yolov8n, 2-class)

Floating-litter + water-hyacinth YOLO detector for a USV on Bengaluru lakes.
Runs the two controlled runs from `kaggle/RUN_PLAN.md` on the verified 2-class
dataset (`data/processed/merged2`, classes `0=litter`, `1=hyacinth`).

**Locked settings (do not tune):** `seed=42`, `epochs=100`, `patience=20`,
`imgsz=960` (D11 box-size audit), `deterministic=True`.

| Run | Init | Notes |
|-----|------|-------|
| A | COCO (`yolov8n.pt`) | baseline |
| B | Zenodo `Model_tiles_weights_converted.pt` (nc=3) | Ultralytics auto-transfers **349/355** state-dict keys into the nc=2 model, skipping exactly the 6 `model.22.*` Detect cls-conv keys (verified, see RUN_PLAN.md) |

## Colab caveats (from `kaggle/RUN_PLAN.md` §Colab caveat)

- Whether 100 epochs at imgsz 960 fits a **free Colab session** (RAM and the
  ~12 h session limit) is **unverified** — check before launching the full P1 run.
  If the session dies, re-run from the last checkpoint cell-by-cell.
- **P2 note:** the val split is *same-site* and will flatter the model. The
  Bengaluru Capture Set (T9) is the real test — score it BEFORE any fine-tune on it.
- License: Ultralytics-pretrained weights (incl. `yolov8n.pt` and anything derived
  from the Zenodo checkpoints) are **AGPL-3.0** — see RUN_PLAN.md §License note.

Run the cells top-to-bottom. Every cell is small and re-runnable.

In [ ]:
# 1. Setup
%pip install -q ultralytics

import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > Hardware accelerator > T4 GPU.")

## 2. Data

Upload the Kaggle-style dataset zip produced by `scripts/build_kaggle_bundle.py`
(`roboat-processed.zip`, top level: `merged2/ lists/ yamls/
zenodo_12800597/converted/` + `BUNDLE_README.txt` — layout per
`kaggle/README.md` §1; the `ood_aquatrash` OOD tree is deliberately not
bundled).

**Option A (recommended):** put `roboat-processed.zip` in your Drive root
`MyDrive/`, then run the next cell.

**Option B:** skip Drive and use the upload cell after it (~3.7k FML + 1.2k
TUD-GV + 1.2k Hagenbeek JPEGs; the upload can take a while).

In [ ]:
# 2a. Data from Google Drive (Option A)
USE_DRIVE = True  # set False if you use the upload cell instead

from pathlib import Path
DATA_ROOT = Path("/content/roboat-processed")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = Path("/content/drive/MyDrive/roboat-processed.zip")
    assert zip_path.exists(), f"{zip_path} not found - upload it to MyDrive first"
    !unzip -q -o {zip_path} -d /content/roboat-processed/

for d in ("merged2", "lists", "yamls", "zenodo_12800597/converted"):
    p = DATA_ROOT / d
    assert p.exists(), f"missing {p} - wrong zip?"
print("dataset OK at", DATA_ROOT)
print("train list lines:", len((DATA_ROOT / "lists/train.txt").read_text().splitlines()))

In [ ]:
# 2b. Data by direct upload (Option B) - only if USE_DRIVE was False
USE_UPLOAD = not USE_DRIVE

from pathlib import Path
DATA_ROOT = Path("/content/roboat-processed")

if USE_UPLOAD:
    from google.colab import files
    up = files.upload()  # pick roboat-processed.zip
    !unzip -q -o /content/roboat-processed.zip -d /content/roboat-processed/

for d in ("merged2", "lists", "yamls", "zenodo_12800597/converted"):
    p = DATA_ROOT / d
    assert p.exists(), f"missing {p} - wrong zip?"
print("dataset OK at", DATA_ROOT)

In [ ]:
# 2c. Rebase yaml/list paths onto the Colab mount point.
# Port of rebase() from kaggle/train_baseline.py:27-58: build_dataset.py writes
# ROOT-relative paths ("data/processed/merged2/..."), so map whatever follows
# "data/processed/" onto DATA_ROOT. Yaml path/train/val/test get the same
# treatment (the shipped yamls carry absolute C:/Users/... paths, which the
# marker also catches).
import yaml
from pathlib import Path

def rebase(data_root: Path, workdir: Path):
    lists = workdir / "lists"
    yamls = workdir / "yamls"
    lists.mkdir(parents=True, exist_ok=True)
    yamls.mkdir(parents=True, exist_ok=True)
    marker = "data/processed/"
    for f in (data_root / "lists").glob("*.txt"):
        out = []
        for line in f.read_text().splitlines():
            if marker in line:
                out.append(f"{data_root.as_posix()}/{line.split(marker)[-1]}")
            elif line:
                out.append(f"{data_root.as_posix()}/{line}")
        (lists / f.name).write_text("\n".join(out) + "\n")
    for f in (data_root / "yamls").glob("*.yaml"):
        y = yaml.safe_load(f.read_text())
        for key in ("train", "val", "test"):
            v = y.get(key)
            if v:
                sv = str(v).replace("\\", "/")
                y[key] = (f"{data_root.as_posix()}/{sv.split(marker)[-1]}"
                          if marker in sv else
                          (f"{data_root.as_posix()}/{sv}" if sv.startswith("data/") else sv))
        y["path"] = data_root.as_posix()
        (yamls / f.name).write_text(yaml.safe_dump(y))
    return yamls

REBASED = rebase(DATA_ROOT, Path("/content/rebased"))
TRAIN_YAML = REBASED / "combined_c2.yaml"  # 2-class: 0=litter, 1=hyacinth
print(TRAIN_YAML.read_text())

## 3. Run A — yolov8n, COCO init

`yolov8n.pt` stock COCO weights, trained on `combined_c2` with the locked
settings. ~100 epochs at imgsz 960 on a T4 exceeds the free session limit
likely — ultralytics resumes are not automatic here; if the session dies,
restart and pass `resume=True` with the run's `last.pt`.

In [ ]:
# 3. Run A (COCO init)
from ultralytics import YOLO

RUNS = "/content/roboat_runs"
model_a = YOLO("yolov8n.pt")  # COCO init
model_a.train(data=str(TRAIN_YAML), epochs=100, patience=20, imgsz=960,
              seed=42, deterministic=True,
              project=RUNS, name="runA_coco_init")
best_A = model_a.trainer.best
print("Run A best:", best_A)

## 4. Run B — yolov8n, init from converted Model_tiles checkpoint

Starts from
`zenodo_12800597/converted/Model_tiles_weights_converted.pt` (nc=3:
`ff_litter/hyacinth/ent_litter`, converted with the `YOLO()` loader by
`scripts/convert_zenodo_weights.py` — never raw `torch.load`). Ultralytics
transfers the compatible layers automatically: **349/355** state-dict keys
into the nc=2 model, skipping exactly the 6 `model.22.*` Detect cls-conv
keys. No manual surgery needed. The converted checkpoint is a yolov8n
architecture, so Run B is the yolov8n leg only.

In [ ]:
# 4. Run B (Zenodo Model_tiles init)
from ultralytics import YOLO

CKPT = DATA_ROOT / "zenodo_12800597/converted/Model_tiles_weights_converted.pt"
assert CKPT.exists(), f"{CKPT} missing - is it in the bundle zip?"

model_b = YOLO(str(CKPT))  # YOLO loader only - never raw torch.load
model_b.train(data=str(TRAIN_YAML), epochs=100, patience=20, imgsz=960,
              seed=42, deterministic=True,
              project=RUNS, name="runB_zenodo_init")
best_B = model_b.trainer.best
print("Run B best:", best_B)

## 5. Eval — P1 gate: per-class AND per-source val mAP@0.5

The P1 gate (RUN_PLAN.md) requires val mAP50 reported **per class** and
**per source** (`saigon_tiles_c2`, `hagenbeek_tiles_c2`), never one
aggregate. This cell validates each run's `best.pt` on both per-source
yamls (plus `combined_c2` for reference) and prints the two tables, then
saves everything to a results JSON.

In [ ]:
# 5. Per-source + per-class val (P1 gate)
import json
from pathlib import Path
from ultralytics import YOLO

PER_SOURCE = ["saigon_tiles_c2", "hagenbeek_tiles_c2", "combined_c2"]

def map50_row(metrics):
    """Aggregate + per-class mAP@0.50 (port of kaggle/train_baseline.py:61-68)."""
    names = metrics.names
    idx = metrics.box.ap_class_index
    ap50 = metrics.box.ap50
    per_class = ({names[int(c)]: float(ap) for c, ap in zip(idx, ap50)}
                 if idx is not None and len(idx) else {})
    return {"all": float(metrics.box.map50), "per_class": per_class}

runs = {}
if Path(best_A).exists(): runs["runA_coco_init"] = best_A
if Path(best_B).exists(): runs["runB_zenodo_init"] = best_B

results = {}
for rname, best in runs.items():
    results[rname] = {}
    for ys in PER_SOURCE:
        m = YOLO(best)
        metrics = m.val(data=str(REBASED / f"{ys}.yaml"), imgsz=960, split="val")
        results[rname][ys] = map50_row(metrics)

# aggregate per-source table
print("===== val mAP@0.5 per source (aggregate) =====")
print(f"{'run':<20}" + "".join(f"{y:<22}" for y in PER_SOURCE))
for rname, row in results.items():
    print(f"{rname:<20}" + "".join(f"{row[y]['all']:<22.4f}" for y in PER_SOURCE))

# per-class per-source table (the P1 gate)
print("\n===== per-class val mAP@0.5 per source (P1 gate) =====")
for rname, row in results.items():
    for ys, vals in row.items():
        pc = "  ".join(f"{cls}={ap:.4f}" for cls, ap in vals["per_class"].items())
        print(f"{rname:<20}{ys:<22}{pc}")

json_path = Path(RUNS) / "p1_gate_results.json"
json_path.write_text(json.dumps(results, indent=2))
print("\nsaved:", json_path)

## 6. Save checkpoints to Drive

Copies each run's `best.pt` (and the P1 results JSON) into
`MyDrive/roboat_runs/`. Colab VMs are ephemeral — do this before the
session ends.

In [ ]:
# 6. Copy best.pt + results JSON to Drive
import shutil
from pathlib import Path

dst = Path("/content/drive/MyDrive/roboat_runs")
if not dst.parent.exists():
    from google.colab import drive
    drive.mount("/content/drive")
dst.mkdir(parents=True, exist_ok=True)

for rname, best in runs.items():
    shutil.copy2(best, dst / f"{rname}_best.pt")
    print("saved:", dst / f"{rname}_best.pt")
shutil.copy2(json_path, dst / "p1_gate_results.json")
print("saved:", dst / "p1_gate_results.json")